<a href="https://colab.research.google.com/github/Of-Calls/sisicallcall-verification-finetuning/blob/main/titanet_verification_eval_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TitaNet Verification Evaluation Colab v5 — Drive 마운트 경로 고정 버전

이 노트북은 Google Drive API file ID 다운로드를 쓰지 않습니다.

구조:
- Google Drive 전체 마운트
- `MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터` 폴더를 이름 기준으로 찾음
- 그 안의 `titanet_local_subset_small.zip` 사용
- 그 안의 `titanet_finetune_runs/titanet_20260427_032738/titanet_small/titanet_small_finetuned_final.nemo` 사용
- 결과는 같은 `화자검증 데이터/titanet_eval_runs/{RUN_ID}`에 저장

경로 유니코드 문제를 피하려고, 문자열 전체 경로를 하드코딩하지 않고 폴더명 component를 NFC/NFD 정규화해서 찾습니다.


In [ ]:

# =========================
# 0. Mount Google Drive 전체
# =========================
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

MYDRIVE = Path("/content/drive/MyDrive")
assert MYDRIVE.exists(), "MyDrive mount failed"

print("MYDRIVE:", MYDRIVE)
print("Top-level:")
for p in sorted(MYDRIVE.iterdir()):
    print(" -", p.name)


Mounted at /content/drive
MYDRIVE: /content/drive/MyDrive
Top-level:
 - AI컴퓨터비전 프로젝트
 - ai 스터디
 - github-recovery-codes.txt
 - python_1900_kdy
 - sqld
 - 랭체인 AI 영상객체탐지분석 플랫폼 구축
 - 집중도 체크 프로그램
 - 취업 관련


In [ ]:

# =========================
# 1. Config: 실제 Drive 폴더 구조 기준
# =========================
from pathlib import Path
from datetime import datetime
import unicodedata

# 네 Drive 구조
PROJECT_FOLDER_NAME = "랭체인 AI 영상객체탐지분석 플랫폼 구축"
SERVICE_FOLDER_NAME = "오브콜스(Of-Calls)"
DATA_FOLDER_NAME = "화자검증 데이터"

DATA_ZIP_NAME = "titanet_local_subset_small.zip"

FINETUNE_RUN_NAME = "titanet_20260427_032738"
FINETUNED_NEMO_RELATIVE = Path("titanet_finetune_runs") / FINETUNE_RUN_NAME / "titanet_small" / "titanet_small_finetuned_final.nemo"

BASELINE_MODEL_NAME = "titanet_small"

# 처음엔 enroll_sec별 1000개만 평가. 전체 30000 trial 평가하려면 0.
MAX_TRIALS_PER_ENROLL_SEC = 0

SEED = 42
TARGET_FARS = [0.01, 0.05, 0.10]
RUN_ID = datetime.now().strftime("eval_%Y%m%d_%H%M%S")

print("PROJECT_FOLDER_NAME:", PROJECT_FOLDER_NAME)
print("SERVICE_FOLDER_NAME:", SERVICE_FOLDER_NAME)
print("DATA_FOLDER_NAME:", DATA_FOLDER_NAME)
print("DATA_ZIP_NAME:", DATA_ZIP_NAME)
print("FINETUNED_NEMO_RELATIVE:", FINETUNED_NEMO_RELATIVE)
print("RUN_ID:", RUN_ID)


PROJECT_FOLDER_NAME: 랭체인 AI 영상객체탐지분석 플랫폼 구축
SERVICE_FOLDER_NAME: 오브콜스(Of-Calls)
DATA_FOLDER_NAME: 화자검증 데이터
DATA_ZIP_NAME: titanet_local_subset_small.zip
FINETUNED_NEMO_RELATIVE: titanet_finetune_runs/titanet_20260427_032738/titanet_small/titanet_small_finetuned_final.nemo
RUN_ID: eval_20260427_053001


In [ ]:

# =========================
# 2. Resolve Drive path by normalized folder names
# =========================
def norm_name(s):
    return unicodedata.normalize("NFC", str(s)).strip()

def find_child_dir(parent: Path, target_name: str) -> Path:
    target = norm_name(target_name)
    children = list(parent.iterdir())

    # exact normalized match
    for child in children:
        if child.is_dir() and norm_name(child.name) == target:
            return child

    # debug output
    print(f"[NOT FOUND] target dir: {target_name}")
    print(f"children of {parent}:")
    for child in children:
        if child.is_dir():
            print(" -", repr(child.name), "normalized:", repr(norm_name(child.name)))
    raise FileNotFoundError(f"Cannot find folder {target_name} under {parent}")

PROJECT_DIR = find_child_dir(MYDRIVE, PROJECT_FOLDER_NAME)
SERVICE_DIR = find_child_dir(PROJECT_DIR, SERVICE_FOLDER_NAME)
DRIVE_DATA_DIR = find_child_dir(SERVICE_DIR, DATA_FOLDER_NAME)

DATA_ZIP_PATH = DRIVE_DATA_DIR / DATA_ZIP_NAME
FINETUNED_NEMO_PATH = DRIVE_DATA_DIR / FINETUNED_NEMO_RELATIVE

EVAL_SAVE_DIR = DRIVE_DATA_DIR / "titanet_eval_runs" / RUN_ID
EVAL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("SERVICE_DIR:", SERVICE_DIR)
print("DRIVE_DATA_DIR:", DRIVE_DATA_DIR)
print("DATA_ZIP_PATH:", DATA_ZIP_PATH)
print("DATA_ZIP exists:", DATA_ZIP_PATH.exists())
print("FINETUNED_NEMO_PATH:", FINETUNED_NEMO_PATH)
print("FINETUNED_NEMO exists:", FINETUNED_NEMO_PATH.exists())
print("EVAL_SAVE_DIR:", EVAL_SAVE_DIR)

print("\nFiles in DRIVE_DATA_DIR:")
for p in sorted(DRIVE_DATA_DIR.iterdir()):
    print(" -", p.name)

assert DATA_ZIP_PATH.exists(), f"Dataset zip not found: {DATA_ZIP_PATH}"
assert FINETUNED_NEMO_PATH.exists(), f"Fine-tuned nemo not found: {FINETUNED_NEMO_PATH}"


PROJECT_DIR: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축
SERVICE_DIR: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)
DRIVE_DATA_DIR: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터
DATA_ZIP_PATH: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_local_subset_small.zip
DATA_ZIP exists: True
FINETUNED_NEMO_PATH: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small/titanet_small_finetuned_final.nemo
FINETUNED_NEMO exists: True
EVAL_SAVE_DIR: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_eval_runs/eval_20260427_053001

Files in DRIVE_DATA_DIR:
 - titanet_eval_runs
 - titanet_finetune_colab_v3.

In [ ]:

# =========================
# 3. Optional install / repair
# =========================
# NeMo / numpy import 에러가 나면 True로 바꾸고 실행 후 런타임 재시작
INSTALL_OR_REPAIR = False

if INSTALL_OR_REPAIR:
    !pip install -q "nemo_toolkit[asr]"
    !pip uninstall -y numpy
    !pip install --no-cache-dir --force-reinstall "numpy==1.26.4"
    print("설치/복구 완료. 런타임 재시작 후 처음부터 다시 실행하세요.")
else:
    print("Skip install/repair.")


Skip install/repair.


In [ ]:

# =========================
# 4. Imports
# =========================
import os
import sys
import json
import ast
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import soundfile as sf
from tqdm.auto import tqdm

import lightning.pytorch as pl
from nemo.collections.asr.models import EncDecSpeakerLabelModel

print("python:", sys.version)
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("lightning.pytorch:", pl.__version__)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
numpy: 1.26.4
torch: 2.10.0+cu128
cuda: True
gpu: NVIDIA A100-SXM4-40GB
lightning.pytorch: 2.4.0


In [ ]:

# =========================
# 5. Copy ZIP to /content and extract
# =========================
WORK_ZIP = Path("/content/titanet_local_subset_small.zip")
EXTRACT_ROOT = Path("/content/titanet_eval_extract")

if EXTRACT_ROOT.exists():
    print("Remove old extract root:", EXTRACT_ROOT)
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

if not WORK_ZIP.exists() or WORK_ZIP.stat().st_size != DATA_ZIP_PATH.stat().st_size:
    print("Copy dataset ZIP to /content ...")
    shutil.copy2(DATA_ZIP_PATH, WORK_ZIP)
else:
    print("Dataset ZIP already copied:", WORK_ZIP)

with zipfile.ZipFile(WORK_ZIP, "r") as zf:
    names = zf.namelist()
    print("ZIP first 30 entries:")
    for n in names[:30]:
        print(" -", n)
    print("Extract to:", EXTRACT_ROOT)
    zf.extractall(EXTRACT_ROOT)

# /content 전체 탐색 금지. EXTRACT_ROOT 내부만 탐색.
candidates = []
if (EXTRACT_ROOT / "manifests").is_dir() and (EXTRACT_ROOT / "wavs").is_dir():
    candidates.append(EXTRACT_ROOT)

for p in EXTRACT_ROOT.rglob("*"):
    if not p.is_dir():
        continue
    try:
        if (p / "manifests").is_dir() and (p / "wavs").is_dir():
            candidates.append(p)
    except OSError:
        continue

candidates = sorted(set(candidates), key=lambda x: len(str(x)))

print("Dataset candidates:")
for c in candidates:
    print(" -", c)

if not candidates:
    print("Extract root children:")
    for p in EXTRACT_ROOT.iterdir():
        print(" -", p)
    raise FileNotFoundError("Could not find dataset dir with manifests/ and wavs/.")

DATA_DIR = candidates[0]
MANIFEST_DIR = DATA_DIR / "manifests"
COLAB_MANIFEST_DIR = DATA_DIR / "manifests_colab"
COLAB_MANIFEST_DIR.mkdir(exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("MANIFEST_DIR:", MANIFEST_DIR)
print("COLAB_MANIFEST_DIR:", COLAB_MANIFEST_DIR)
print("Manifest files:", [p.name for p in MANIFEST_DIR.iterdir()][:30])


Remove old extract root: /content/titanet_eval_extract
Dataset ZIP already copied: /content/titanet_local_subset_small.zip
ZIP first 30 entries:
 - manifests/
 - manifests/enrollment_manifest_local.csv
 - manifests/eval_manifest.json
 - manifests/evaluation_verification_manifest_local.csv
 - manifests/train_manifest.json
 - manifests/trials_local.csv
 - manifests/valid_manifest.json
 - README_TITANET_LOCAL_SUBSET.md
 - reports/
 - reports/audio_check_report.csv
 - reports/export_summary.json
 - reports/export_titanet_local_subset.log
 - reports/extracted_files.csv
 - reports/failed_extracts.csv
 - wavs/
 - wavs/eval/
 - wavs/eval/0000/
 - wavs/eval/0000/A0002-0000F1012-10120120-00000096_d2a8c8348e.wav
 - wavs/eval/0000/A0009-0000F1012-10120120-00367759_2a12669044.wav
 - wavs/eval/0000/A0010-0000F1012-10120120-00368091_2126cad9a4.wav
 - wavs/eval/0000/A0013-0000F1012-10120120-00368770_e6cd8d4fc3.wav
 - wavs/eval/0000/A0014-0000F1012-10120130-00369282_4b27a49007.wav
 - wavs/eval/0000/A00

In [ ]:

# =========================
# 6. Rewrite trials paths to /content paths
# =========================
LOCAL_ROOT_MARKERS = [
    "D:/aihub_check/titanet_local_subset_small",
    r"D:\aihub_check\titanet_local_subset_small",
]
COLAB_ROOT_STR = str(DATA_DIR).replace("\\", "/")


def rewrite_path_to_colab(path_str):
    if path_str is None:
        return path_str

    s = str(path_str).replace("\\", "/")

    for marker in LOCAL_ROOT_MARKERS:
        m = marker.replace("\\", "/")
        if s.startswith(m):
            return s.replace(m, COLAB_ROOT_STR, 1)

    if s.startswith("/content/"):
        return s

    idx = s.find("/wavs/")
    if idx >= 0:
        return COLAB_ROOT_STR + s[idx:]

    return s


def parse_audio_refs(x):
    if pd.isna(x):
        return []
    s = str(x)
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, list):
            return [str(v) for v in obj]
    except Exception:
        pass
    if ";" in s:
        return [v for v in s.split(";") if v]
    return [s]


def rewrite_audio_refs_field(x):
    refs = parse_audio_refs(x)
    return str([rewrite_path_to_colab(v) for v in refs])


def rewrite_trials_to_colab(src, dst):
    df = pd.read_csv(src)
    required = ["verification_audio_ref", "enrollment_audio_refs", "enroll_sec", "label"]
    missing = [c for c in required if c not in df.columns]
    assert not missing, f"Missing trial columns: {missing}"

    df["verification_audio_ref"] = df["verification_audio_ref"].map(rewrite_path_to_colab)
    df["enrollment_audio_refs"] = df["enrollment_audio_refs"].map(rewrite_audio_refs_field)
    df.to_csv(dst, index=False)
    return df

trial_candidates = [
    MANIFEST_DIR / "trials_local.csv",
    MANIFEST_DIR / "trials_colab.csv",
    COLAB_MANIFEST_DIR / "trials_colab.csv",
]

trials_src = None
for p in trial_candidates:
    if p.exists():
        trials_src = p
        break

assert trials_src is not None, f"No trials file found in {trial_candidates}"

TRIALS_COLAB = COLAB_MANIFEST_DIR / "trials_colab.csv"
trials_df = rewrite_trials_to_colab(trials_src, TRIALS_COLAB)

print("trials_src:", trials_src)
print("TRIALS_COLAB:", TRIALS_COLAB)
print("trials rows:", len(trials_df))
print(trials_df.groupby(["enroll_sec", "label"]).size())


trials_src: /content/titanet_eval_extract/manifests/trials_local.csv
TRIALS_COLAB: /content/titanet_eval_extract/manifests_colab/trials_colab.csv
trials rows: 30000
enroll_sec  label
1.0         0        5687
            1        1900
2.0         0        5608
            1        1868
3.0         0        5577
            1        1857
5.0         0        5628
            1        1875
dtype: int64


In [ ]:

# =========================
# 7. Sample trials
# =========================
df = pd.read_csv(TRIALS_COLAB)
df["enroll_sec"] = df["enroll_sec"].astype(float)
df["label"] = df["label"].astype(int)

if MAX_TRIALS_PER_ENROLL_SEC and MAX_TRIALS_PER_ENROLL_SEC > 0:
    sampled_parts = []

    for enroll_sec, g in df.groupby("enroll_sec"):
        n_total = min(len(g), MAX_TRIALS_PER_ENROLL_SEC)
        label_ratio = g["label"].value_counts(normalize=True).to_dict()
        sub_parts = []
        used = 0

        for label_value in sorted(g["label"].unique()):
            gg = g[g["label"] == label_value]
            n = int(round(n_total * label_ratio.get(label_value, 0)))
            n = min(n, len(gg))
            if n > 0:
                sub_parts.append(gg.sample(n=n, random_state=SEED + int(enroll_sec * 10) + label_value))
                used += n

        if used < n_total:
            used_idx = pd.concat(sub_parts).index if sub_parts else []
            rest = g.drop(index=used_idx, errors="ignore")
            if len(rest) > 0:
                sub_parts.append(rest.sample(n=min(n_total - used, len(rest)), random_state=SEED))

        sampled_parts.append(pd.concat(sub_parts))

    eval_trials = pd.concat(sampled_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
else:
    eval_trials = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

EVAL_TRIALS_PATH = EVAL_SAVE_DIR / "eval_trials_used.csv"
eval_trials.to_csv(EVAL_TRIALS_PATH, index=False)

print("eval_trials:", len(eval_trials))
print(eval_trials.groupby(["enroll_sec", "label"]).size())
print("saved:", EVAL_TRIALS_PATH)


eval_trials: 30000
enroll_sec  label
1.0         0        5687
            1        1900
2.0         0        5608
            1        1868
3.0         0        5577
            1        1857
5.0         0        5628
            1        1875
dtype: int64
saved: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_eval_runs/eval_20260427_053001/eval_trials_used.csv


In [ ]:

# =========================
# 8. Load baseline / fine-tuned models
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_baseline():
    model = EncDecSpeakerLabelModel.from_pretrained(BASELINE_MODEL_NAME)
    model.eval()
    model.to(device)
    return model

def load_finetuned():
    model = EncDecSpeakerLabelModel.restore_from(str(FINETUNED_NEMO_PATH))
    model.eval()
    model.to(device)
    return model

baseline_model = load_baseline()
finetuned_model = load_finetuned()

print("baseline loaded:", type(baseline_model))
print("finetuned loaded:", type(finetuned_model))
print("FINETUNED_NEMO_PATH:", FINETUNED_NEMO_PATH)


[NeMo I 2026-04-27 05:30:27 cloud:58] Found existing object /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-s/908e9576cf7dd7420e75f73ceb0b72e1/titanet-s.nemo.
[NeMo I 2026-04-27 05:30:27 cloud:64] Re-using file from: /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-s/908e9576cf7dd7420e75f73ceb0b72e1/titanet-s.nemo
[NeMo I 2026-04-27 05:30:27 common:939] Instantiating model from pre-trained checkpoint


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
    - id07182
    - id07183
    - id07185
    - id07186
    - id07187
    - id07188
    - id07189
    - id07191
    - id07192
    - id07194
    - id07195
    - id07196
    - id07197
    - id07198
    - id07199
    - id07200
    - id07202
    - id07204
    - id07205
    - id07206
    - id07207
    - id07208
    - id07209
    - id07210
    - id07212
    - id07213
    - id07214
    - id07215
    - id07217
    - id07218
    - id07219
    - id07220
    - id07221
    - id07223
    - id07227
    - id07228
    - id07229
    - id07230
    - id07232
    - id07233
    - id07234
    - id07235
    - id07236
    - id07238
    - id07240
    - id07241
    - id07242
    - id07243
    - id07244
    - id07246
    - id07247
    - id07250
    - id07251
    - id07253
    - id07254
    - id07255
    - id07256
    - id07258
    - id07259
    - id07262
    - id07263
    - id07264
    - id07265
    - id07268
    - id07269
    - id07272
    - id07273
    - id07275
    - id0727

[NeMo I 2026-04-27 05:30:32 save_restore_connector:285] Model EncDecSpeakerLabelModel was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-s/908e9576cf7dd7420e75f73ceb0b72e1/titanet-s.nemo.


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
    - id07182
    - id07183
    - id07185
    - id07186
    - id07187
    - id07188
    - id07189
    - id07191
    - id07192
    - id07194
    - id07195
    - id07196
    - id07197
    - id07198
    - id07199
    - id07200
    - id07202
    - id07204
    - id07205
    - id07206
    - id07207
    - id07208
    - id07209
    - id07210
    - id07212
    - id07213
    - id07214
    - id07215
    - id07217
    - id07218
    - id07219
    - id07220
    - id07221
    - id07223
    - id07227
    - id07228
    - id07229
    - id07230
    - id07232
    - id07233
    - id07234
    - id07235
    - id07236
    - id07238
    - id07240
    - id07241
    - id07242
    - id07243
    - id07244
    - id07246
    - id07247
    - id07250
    - id07251
    - id07253
    - id07254
    - id07255
    - id07256
    - id07258
    - id07259
    - id07262
    - id07263
    - id07264
    - id07265
    - id07268
    - id07269
    - id07272
    - id07273
    - id07275
    - id0727

[NeMo I 2026-04-27 05:30:38 save_restore_connector:285] Model EncDecSpeakerLabelModel was successfully restored from /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small/titanet_small_finetuned_final.nemo.
baseline loaded: <class 'nemo.collections.asr.models.label_models.EncDecSpeakerLabelModel'>
finetuned loaded: <class 'nemo.collections.asr.models.label_models.EncDecSpeakerLabelModel'>
FINETUNED_NEMO_PATH: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small/titanet_small_finetuned_final.nemo


In [ ]:

# =========================
# 9. Embedding helpers
# =========================
def extract_embedding(model, audio_path):
    audio_path = str(audio_path)
    last_err = None

    if hasattr(model, "get_embedding"):
        try:
            emb = model.get_embedding(audio_path)
            if isinstance(emb, torch.Tensor):
                emb = emb.detach().cpu().float().numpy()
            return np.asarray(emb).squeeze()
        except Exception as e:
            last_err = e

    if hasattr(model, "infer_file"):
        try:
            out = model.infer_file(audio_path)
            if isinstance(out, tuple):
                for item in out:
                    arr = np.asarray(item)
                    if arr.size > 10:
                        return arr.squeeze()
            return np.asarray(out).squeeze()
        except Exception as e:
            last_err = e

    raise RuntimeError(f"Could not extract embedding. path={audio_path}, err={repr(last_err)}")


def normalize_embedding(x):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    return x / (np.linalg.norm(x) + 1e-12)


def collect_unique_audio_paths(trials):
    paths = set()
    for _, row in trials.iterrows():
        paths.add(rewrite_path_to_colab(row["verification_audio_ref"]))
        for ref in parse_audio_refs(row["enrollment_audio_refs"]):
            paths.add(rewrite_path_to_colab(ref))

    paths = sorted(paths)
    missing = [p for p in paths if not Path(p).exists()]
    if missing:
        print("Missing sample:")
        for p in missing[:20]:
            print(p)
        raise FileNotFoundError(f"Missing {len(missing)} audio files")

    return paths

unique_audio_paths = collect_unique_audio_paths(eval_trials)
print("unique_audio_paths:", len(unique_audio_paths))
print(unique_audio_paths[:3])


unique_audio_paths: 3221
['/content/titanet_eval_extract/wavs/eval/0000/A0002-0000F1012-10120120-00000096_d2a8c8348e.wav', '/content/titanet_eval_extract/wavs/eval/0000/A0009-0000F1012-10120120-00367759_2a12669044.wav', '/content/titanet_eval_extract/wavs/eval/0000/A0010-0000F1012-10120120-00368091_2126cad9a4.wav']


In [ ]:

# =========================
# 10. Extract embeddings
# =========================
def extract_embeddings_for_model(model, model_tag, audio_paths):
    cache = {}
    failures = []

    for p in tqdm(audio_paths, desc=f"extract {model_tag}"):
        try:
            emb = extract_embedding(model, p)
            cache[p] = normalize_embedding(emb)
        except Exception as e:
            failures.append({"audio_path": p, "error": repr(e)})

    fail_path = EVAL_SAVE_DIR / f"{model_tag}_embedding_failures.csv"
    pd.DataFrame(failures).to_csv(fail_path, index=False)

    print(model_tag, "success:", len(cache), "failures:", len(failures), "fail_path:", fail_path)

    if not cache:
        raise RuntimeError(f"No embeddings extracted for {model_tag}")

    return cache, failures

baseline_embs, baseline_failures = extract_embeddings_for_model(baseline_model, "baseline", unique_audio_paths)
finetuned_embs, finetuned_failures = extract_embeddings_for_model(finetuned_model, "finetuned", unique_audio_paths)


extract baseline:   0%|          | 0/3221 [00:00<?, ?it/s]

baseline success: 3221 failures: 0 fail_path: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_eval_runs/eval_20260427_053001/baseline_embedding_failures.csv


extract finetuned:   0%|          | 0/3221 [00:00<?, ?it/s]

finetuned success: 3221 failures: 0 fail_path: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_eval_runs/eval_20260427_053001/finetuned_embedding_failures.csv


In [ ]:

# =========================
# 11. Score trials
# =========================
def score_trials(trials, emb_cache):
    rows = []
    skipped = 0

    for _, row in tqdm(trials.iterrows(), total=len(trials), desc="score trials"):
        verification_path = rewrite_path_to_colab(row["verification_audio_ref"])
        enroll_paths = [rewrite_path_to_colab(x) for x in parse_audio_refs(row["enrollment_audio_refs"])]

        if verification_path not in emb_cache:
            skipped += 1
            continue

        enroll_vecs = [emb_cache[p] for p in enroll_paths if p in emb_cache]
        if not enroll_vecs:
            skipped += 1
            continue

        enroll_emb = normalize_embedding(np.mean(np.stack(enroll_vecs, axis=0), axis=0))
        ver_emb = emb_cache[verification_path]
        score = float(np.dot(enroll_emb, ver_emb))

        rows.append({
            "trial_id": row.get("trial_id", ""),
            "split": row.get("split", ""),
            "enroll_sec": float(row["enroll_sec"]),
            "label": int(row["label"]),
            "trial_type": row.get("trial_type", ""),
            "score": score,
            "enroll_speaker_id": row.get("enroll_speaker_id", ""),
            "test_speaker_id": row.get("test_speaker_id", ""),
        })

    return pd.DataFrame(rows), skipped

baseline_scores, baseline_skipped = score_trials(eval_trials, baseline_embs)
finetuned_scores, finetuned_skipped = score_trials(eval_trials, finetuned_embs)

baseline_scores["model"] = "baseline"
finetuned_scores["model"] = "finetuned"

baseline_scores.to_csv(EVAL_SAVE_DIR / "baseline_trial_scores.csv", index=False)
finetuned_scores.to_csv(EVAL_SAVE_DIR / "finetuned_trial_scores.csv", index=False)

print("baseline_scores:", baseline_scores.shape, "skipped:", baseline_skipped)
print("finetuned_scores:", finetuned_scores.shape, "skipped:", finetuned_skipped)


score trials:   0%|          | 0/30000 [00:00<?, ?it/s]

score trials:   0%|          | 0/30000 [00:00<?, ?it/s]

baseline_scores: (30000, 9) skipped: 0
finetuned_scores: (30000, 9) skipped: 0


In [ ]:

# =========================
# 12. Metrics
# =========================
def compute_eer(labels, scores):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)

    thresholds = np.unique(scores)
    thresholds = np.concatenate([[scores.max() + 1e-6], thresholds[::-1], [scores.min() - 1e-6]])

    pos = labels == 1
    neg = labels == 0
    n_pos = max(pos.sum(), 1)
    n_neg = max(neg.sum(), 1)

    fars = []
    frrs = []

    for th in thresholds:
        pred_pos = scores >= th
        fa = np.logical_and(pred_pos, neg).sum()
        fr = np.logical_and(~pred_pos, pos).sum()
        fars.append(fa / n_neg)
        frrs.append(fr / n_pos)

    fars = np.asarray(fars)
    frrs = np.asarray(frrs)

    idx = int(np.argmin(np.abs(fars - frrs)))
    eer = float((fars[idx] + frrs[idx]) / 2.0)
    return eer, float(thresholds[idx])


def tar_at_far(labels, scores, target_far):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)

    thresholds = np.unique(scores)
    thresholds = np.concatenate([[scores.max() + 1e-6], thresholds[::-1], [scores.min() - 1e-6]])

    pos = labels == 1
    neg = labels == 0
    n_pos = max(pos.sum(), 1)
    n_neg = max(neg.sum(), 1)

    best_tar = 0.0
    best_far = 0.0
    best_th = float(thresholds[0])

    for th in thresholds:
        pred_pos = scores >= th
        far = np.logical_and(pred_pos, neg).sum() / n_neg
        tar = np.logical_and(pred_pos, pos).sum() / n_pos
        if far <= target_far and tar >= best_tar:
            best_tar = float(tar)
            best_far = float(far)
            best_th = float(th)

    return best_tar, best_far, best_th


def summarize_scores(df, model_name):
    results = []

    for enroll_sec, g in df.groupby("enroll_sec"):
        labels = g["label"].to_numpy()
        scores = g["score"].to_numpy()
        eer, eer_th = compute_eer(labels, scores)

        row = {
            "model": model_name,
            "enroll_sec": float(enroll_sec),
            "n_trials": int(len(g)),
            "n_pos": int((labels == 1).sum()),
            "n_neg": int((labels == 0).sum()),
            "eer": eer,
            "eer_threshold": eer_th,
            "pos_score_mean": float(scores[labels == 1].mean()) if (labels == 1).any() else None,
            "neg_score_mean": float(scores[labels == 0].mean()) if (labels == 0).any() else None,
        }

        for tfar in TARGET_FARS:
            tar, far, th = tar_at_far(labels, scores, tfar)
            row[f"tar_at_far_{tfar}"] = tar
            row[f"actual_far_at_far_{tfar}"] = far
            row[f"threshold_at_far_{tfar}"] = th

        results.append(row)

    return pd.DataFrame(results)

baseline_summary = summarize_scores(baseline_scores, "baseline")
finetuned_summary = summarize_scores(finetuned_scores, "finetuned")
summary = pd.concat([baseline_summary, finetuned_summary], ignore_index=True)

summary_path = EVAL_SAVE_DIR / "verification_summary_by_enroll_sec.csv"
summary.to_csv(summary_path, index=False)

display(summary)
print("saved:", summary_path)


,model,enroll_sec,n_trials,n_pos,n_neg,eer,eer_threshold,pos_score_mean,neg_score_mean,tar_at_far_0.01,actual_far_at_far_0.01,threshold_at_far_0.01,tar_at_far_0.05,actual_far_at_far_0.05,threshold_at_far_0.05,tar_at_far_0.1,actual_far_at_far_0.1,threshold_at_far_0.1
0,baseline,1.0,7587,1900,5687,0.288399,0.321038,0.399039,0.253393,0.118947,0.009847,0.572986,0.306316,0.049938,0.474161,0.449474,0.099877,0.421987
1,baseline,2.0,7476,1868,5608,0.279433,0.318839,0.400159,0.250757,0.126338,0.009986,0.570719,0.318522,0.049929,0.470260,0.472698,0.099857,0.416157
2,baseline,3.0,7434,1857,5577,0.289738,0.322451,0.398605,0.253062,0.127625,0.009862,0.568739,0.303716,0.049848,0.475195,0.438341,0.099874,0.423816
3,baseline,5.0,7503,1875,5628,0.274149,0.344559,0.425651,0.268098,0.128000,0.009950,0.586673,0.337067,0.049929,0.493363,0.489600,0.099858,0.441671
4,finetuned,1.0,7587,1900,5687,0.253710,0.260367,0.399535,0.076541,0.121053,0.009847,0.638038,0.360526,0.049938,0.492533,0.506316,0.099877,0.416304
5,finetuned,2.0,7476,1868,5608,0.253746,0.259127,0.399109,0.077918,0.126874,0.009986,0.640324,0.350642,0.049929,0.493734,0.494111,0.099857,0.418466
6,finetuned,3.0,7434,1857,5577,0.256907,0.255924,0.398745,0.075658,0.117932,0.009862,0.653445,0.329025,0.049848,0.508584,0.501346,0.099874,0.417058
7,finetuned,5.0,7503,1875,5628,0.233093,0.281895,0.431108,0.079362,0.136000,0.009950,0.655715,0.360533,0.049929,0.522873,0.526933,0.099858,0.436755


saved: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_eval_runs/eval_20260427_053001/verification_summary_by_enroll_sec.csv


In [ ]:

# =========================
# 13. Baseline vs fine-tuned comparison
# =========================
comparison_rows = []

for enroll_sec in sorted(summary["enroll_sec"].unique()):
    b = summary[(summary["model"] == "baseline") & (summary["enroll_sec"] == enroll_sec)].iloc[0]
    f = summary[(summary["model"] == "finetuned") & (summary["enroll_sec"] == enroll_sec)].iloc[0]

    row = {
        "enroll_sec": float(enroll_sec),
        "baseline_eer": float(b["eer"]),
        "finetuned_eer": float(f["eer"]),
        "eer_delta_finetuned_minus_baseline": float(f["eer"] - b["eer"]),
        "eer_improved": bool(f["eer"] < b["eer"]),
    }

    for tfar in TARGET_FARS:
        col = f"tar_at_far_{tfar}"
        row[f"baseline_{col}"] = float(b[col])
        row[f"finetuned_{col}"] = float(f[col])
        row[f"{col}_delta"] = float(f[col] - b[col])

    comparison_rows.append(row)

comparison = pd.DataFrame(comparison_rows)
comparison_path = EVAL_SAVE_DIR / "baseline_vs_finetuned_comparison.csv"
comparison.to_csv(comparison_path, index=False)

display(comparison)
print("saved:", comparison_path)


,enroll_sec,baseline_eer,finetuned_eer,eer_delta_finetuned_minus_baseline,eer_improved,baseline_tar_at_far_0.01,finetuned_tar_at_far_0.01,tar_at_far_0.01_delta,baseline_tar_at_far_0.05,finetuned_tar_at_far_0.05,tar_at_far_0.05_delta,baseline_tar_at_far_0.1,finetuned_tar_at_far_0.1,tar_at_far_0.1_delta
0,1.0,0.288399,0.253710,-0.034689,True,0.118947,0.121053,0.002105,0.306316,0.360526,0.054211,0.449474,0.506316,0.056842
1,2.0,0.279433,0.253746,-0.025687,True,0.126338,0.126874,0.000535,0.318522,0.350642,0.032120,0.472698,0.494111,0.021413
2,3.0,0.289738,0.256907,-0.032831,True,0.127625,0.117932,-0.009693,0.303716,0.329025,0.025310,0.438341,0.501346,0.063005
3,5.0,0.274149,0.233093,-0.041056,True,0.128000,0.136000,0.008000,0.337067,0.360533,0.023467,0.489600,0.526933,0.037333


saved: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_eval_runs/eval_20260427_053001/baseline_vs_finetuned_comparison.csv


In [ ]:

# =========================
# 14. Save metadata and list outputs
# =========================
meta = {
    "run_id": RUN_ID,
    "drive_data_dir": str(DRIVE_DATA_DIR),
    "data_zip_path": str(DATA_ZIP_PATH),
    "finetuned_nemo_path": str(FINETUNED_NEMO_PATH),
    "baseline_model_name": BASELINE_MODEL_NAME,
    "max_trials_per_enroll_sec": MAX_TRIALS_PER_ENROLL_SEC,
    "eval_trials_path": str(EVAL_TRIALS_PATH),
    "n_unique_audio_paths": len(unique_audio_paths),
    "baseline_embedding_failures": len(baseline_failures),
    "finetuned_embedding_failures": len(finetuned_failures),
    "eval_save_dir": str(EVAL_SAVE_DIR),
}
meta_path = EVAL_SAVE_DIR / "eval_meta.json"
meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(meta, ensure_ascii=False, indent=2))
print("\nOutputs:")
for p in sorted(EVAL_SAVE_DIR.rglob("*")):
    if p.is_file():
        print(p)


{
  "run_id": "eval_20260427_053001",
  "drive_data_dir": "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터",
  "data_zip_path": "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_local_subset_small.zip",
  "finetuned_nemo_path": "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small/titanet_small_finetuned_final.nemo",
  "baseline_model_name": "titanet_small",
  "max_trials_per_enroll_sec": 0,
  "eval_trials_path": "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_eval_runs/eval_20260427_053001/eval_trials_used.csv",
  "n_unique_audio_paths": 3221,
  "baseline_embedding_failures": 0,
  "finetuned_embedding_failures": 0,
  "eval_save_dir": "/content/drive/MyDrive/랭체ᄋ